# Structured Output — Getting JSON from LLMs

## Introduction

LLMs generate free-form text, but real applications need **structured data** — dictionaries, typed objects, database rows. Structured output techniques bridge this gap by instructing the model to produce machine-parseable formats (typically JSON) and then validating the result against a schema.

### Why Structured Output Matters

- **Reliable parsing**: Free text is ambiguous; JSON has a clear, deterministic grammar.
- **Type safety**: Pydantic validation catches errors like `"confidence": "very high"` when a float is expected.
- **Data extraction**: Turn unstructured documents into rows of clean, queryable data.
- **Pipeline integration**: Downstream code can consume the output directly without fragile regex hacks.

## What You'll Learn

- How the `OutputParser` class extracts JSON from messy LLM responses
- Using **Pydantic models** to define schemas and validate parsed data
- Generating **format instructions** to include in your prompts
- Handling malformed output with retry strategies

## Learning Resources

- [Pydantic Documentation](https://docs.pydantic.dev/latest/)
- [LangChain Output Parsers](https://python.langchain.com/docs/concepts/output_parsers/)
- [VIDEO: "Pydantic is All You Need"](https://www.youtube.com/watch?v=yj-wSRJwrrc)

In [ ]:
# Import structured output components from AgentExplorr
from agentexplorr.prompt_engineering.structured_output import OutputParser

# We also need Pydantic for defining our data schemas
from pydantic import BaseModel

# Verify imports
print("OutputParser methods:")
for method in ["extract_json", "parse_json", "parse_json_to_model", "parse_with_retry", "get_format_instructions"]:
    print(f"  - {method}")
print("\nImports loaded successfully!")

## The Challenge: LLMs Output Text, Applications Need Data

Consider a real scenario: you ask an LLM to analyze a product review. The LLM might respond with:

> "The sentiment of this review is positive. The reviewer seems quite happy with the product, rating it around 4.5 out of 5. The key topics mentioned are battery life and screen quality."

This is useful for a human, but your code needs to answer:
- What is `sentiment`? (a string: "positive")
- What is `confidence`? (a float: 0.9)
- What are `topics`? (a list: ["battery life", "screen quality"])

**The solution**: Define a Pydantic model describing the schema you want, instruct the LLM to respond in JSON matching that schema, then parse and validate the response with `OutputParser`.

In [ ]:
# ------------------------------------------------------------------
# Define a Pydantic model and parse simulated LLM output
# ------------------------------------------------------------------

# Step 1: Define the schema as a Pydantic model
class ReviewAnalysis(BaseModel):
    """Schema for structured review analysis output."""
    sentiment: str          # "positive", "negative", or "neutral"
    confidence: float       # 0.0 to 1.0
    topics: list[str]       # key topics mentioned in the review
    summary: str            # one-sentence summary

# Step 2: Generate format instructions to include in your prompt
parser = OutputParser()
instructions = parser.get_format_instructions(ReviewAnalysis)
print("=== FORMAT INSTRUCTIONS (include in your prompt) ===")
print(instructions)

# Step 3: Simulate an LLM response (in production, this comes from the API)
simulated_llm_response = """
Here's my analysis of the review:

```json
{
    "sentiment": "positive",
    "confidence": 0.92,
    "topics": ["battery life", "screen quality", "price"],
    "summary": "The reviewer is highly satisfied with the product's battery and display."
}
```

I hope this helps!
"""

# Step 4: Parse and validate the response
result = parser.parse_json_to_model(simulated_llm_response, ReviewAnalysis)

print("\n=== PARSED RESULT ===")
print(f"  Sentiment:  {result.sentiment}")
print(f"  Confidence: {result.confidence}")
print(f"  Topics:     {result.topics}")
print(f"  Summary:    {result.summary}")
print(f"\n  Type: {type(result).__name__} (validated Pydantic model)")

## JSON Extraction and Validation Patterns

The `OutputParser` handles the messy reality of LLM responses using a multi-strategy approach:

### Extraction Strategies (in order of priority)

1. **Markdown code blocks** — LLMs often wrap JSON in ` ```json ... ``` `. The parser detects and extracts from these blocks first.
2. **Brace matching** — If no code block is found, the parser looks for the outermost `{...}` or `[...]` pair in the text.
3. **Retry with cleanup** — `parse_with_retry()` progressively strips common LLM artifacts (e.g., "Here's the output:") before re-attempting extraction.

### Common Patterns for Reliable Output

| Pattern | Description |
|---------|-------------|
| **Include the schema** | Use `get_format_instructions()` to tell the model exactly what JSON fields you expect. |
| **Validate with Pydantic** | Catches type errors (string where float expected) and missing fields immediately. |
| **Retry on failure** | `parse_with_retry()` handles transient formatting issues without crashing your pipeline. |
| **Extract, don't assume** | Never assume the LLM returns only JSON — always use `extract_json()` to handle surrounding text. |

## Key Takeaways

1. **Define your schema first** using Pydantic models. This gives you type safety, validation, and auto-generated format instructions.
2. **Use `OutputParser.extract_json()`** to handle the reality that LLMs wrap JSON in markdown blocks, add explanatory text, or include other artifacts.
3. **`get_format_instructions()`** generates a human-readable schema description you can paste directly into your prompt to guide the model.
4. **`parse_with_retry()`** adds resilience — in production, LLM outputs occasionally vary in format, and retries with progressive cleanup prevent pipeline failures.
5. **Pydantic validation** catches subtle bugs (wrong types, missing fields) that would otherwise propagate silently through your application.

## Next Steps

- **Experiment**: Try defining your own Pydantic models for different extraction tasks (e.g., extracting entities from news articles, parsing meeting notes into action items).
- **Combine with few-shot**: Use the techniques from Notebook 02 to provide examples of the JSON format you want — this significantly improves reliability.
- **Production tip**: In real applications, combine `get_format_instructions()` with a system prompt that says "You must respond with valid JSON only" for the best results.